# 🛡️ AnomalyForensicNet (Clean-Slate From-Scratch Recommendation)
### Dataset: `OwensLab/CommunityForensics` (CVPR 2025) + `defactodataset/defactoinpainting`

Arsitektur yang dirancang khusus dari nol (*clean slate*) untuk mengatasi masalah utama deteksi gabungan **Full AI Synthesis** dan **Local Inpainting**:
1. **Hybrid Fixed + Zero-Mean Constrained Stem + Phase/Magnitude 2D-FFT**: Menekan objek visual sejak layer pertama, mengekstrak 24 kanal residu derau mikro (`SRM/Laplacian/DoG/NPR` + `ZeroMeanConv2d`) serta **Phase Spectrum** (peka terhadap kerusakan fase inpainting).
2. **Mix-Kernel ConvNeXt Backbone (`3x3` + `7x7` Depthwise + `LayerNorm2d`)**: Kernel `3x3` menangkap batas jahitan (*seam boundary*) inpainting yang tajam, sedangkan `7x7` menangkap periodisitas upsampling difusi/GAN tanpa mencampur statistik batch (`LayerNorm2d`).
3. **Intra-Image Patch Inconsistency Attention + Top-K (15%) Anomaly Pooling**: Membandingkan deviasi tiap patch terhadap rata-rata gambar itu sendiri dan mengambil 15% aktivasi tertinggi agar sinyal inpainting lokal tidak tenggelam oleh rata-rata area foto asli.
4. **Dual-Task Supervision (Classification + 14x14 Anomaly Mask Localization)**: Melatih klasifikasi (`FocalLoss`) sekaligus memprediksi peta area manipulasi `14x14` (`BCEWithLogitsLoss`) menggunakan `inpaint_mask` asli dari DEFACTO.

In [ ]:
# @title 🛠️ 1. Persiapan Lingkungan & Hardware Acceleration
import os, io, sys, shutil, time, random, subprocess, importlib.util, warnings
warnings.filterwarnings('ignore')

for pkg in ('pyarrow', 'huggingface_hub', 'kagglehub'):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
warnings.simplefilter('ignore', Image.DecompressionBombWarning)
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, roc_curve

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__} | Device: {device}' + (f' ({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else ''))

In [ ]:
# @title 📦 2. Akuisisi Dataset + Peta Masker Inpainting (CommunityForensics + DEFACTO Inpainting)
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
import kagglehub

VALID_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff')
CF_REAL_SAMPLES = 10000
CF_AI_SAMPLES = 5000
DEFACTO_INPAINT_SAMPLES = 5000

def load_community_forensics(n_real=6000, n_ai=3000):
    CACHE_DIR = 'data/cf_cache'
    os.makedirs(f'{CACHE_DIR}/ai', exist_ok=True)
    os.makedirs(f'{CACHE_DIR}/real', exist_ok=True)
    existing_ai = sorted([os.path.join(f'{CACHE_DIR}/ai', f) for f in os.listdir(f'{CACHE_DIR}/ai') if f.lower().endswith(VALID_EXTS)])
    existing_real = sorted([os.path.join(f'{CACHE_DIR}/real', f) for f in os.listdir(f'{CACHE_DIR}/real') if f.lower().endswith(VALID_EXTS)])
    count_ai, count_real = len(existing_ai), len(existing_real)

    if count_ai < n_ai or count_real < n_real:
        start_shard = min(5, count_real // 4100)
        for shard_path in [f'data/publicEval/HFCF_PublicEval_{i}.parquet' for i in range(start_shard, 6)]:
            if count_ai >= n_ai and count_real >= n_real:
                break
            try:
                local_pq = hf_hub_download(repo_id='OwensLab/CommunityForensics', filename=shard_path, repo_type='dataset')
                pf = pq.ParquetFile(local_pq)
                rg_indices = list(range(pf.num_row_groups))
                random.shuffle(rg_indices)
                for rg_idx in rg_indices:
                    if count_ai >= n_ai and count_real >= n_real:
                        break
                    b_dict = pf.read_row_group(rg_idx).to_pydict()
                    paired = list(zip(b_dict.get('label', []), b_dict.get('image_data', b_dict.get('image', []))))
                    random.shuffle(paired)
                    for lbl_val, img_data in paired:
                        if count_ai >= n_ai and count_real >= n_real:
                            break
                        is_real = (int(lbl_val) == 0)
                        if (is_real and count_real >= n_real) or ((not is_real) and count_ai >= n_ai):
                            continue
                        try:
                            if isinstance(img_data, Image.Image):
                                img = img_data.convert('RGB')
                            elif isinstance(img_data, (bytes, bytearray)):
                                img = Image.open(io.BytesIO(img_data)).convert('RGB')
                            elif isinstance(img_data, dict) and img_data.get('bytes'):
                                img = Image.open(io.BytesIO(img_data['bytes'])).convert('RGB')
                            else:
                                continue
                        except Exception:
                            continue
                        if max(img.size) > 2048:
                            img.thumbnail((2048, 2048), Image.Resampling.BILINEAR)
                        if is_real:
                            save_p = f'{CACHE_DIR}/real/cf_real_{count_real:05d}.jpg'
                            if not os.path.exists(save_p):
                                img.save(save_p, 'JPEG', quality=random.randint(75, 98))
                            existing_real.append(save_p)
                            count_real += 1
                        else:
                            save_p = f'{CACHE_DIR}/ai/cf_ai_{count_ai:05d}.jpg'
                            if not os.path.exists(save_p):
                                img.save(save_p, 'JPEG', quality=random.randint(75, 98))
                            existing_ai.append(save_p)
                            count_ai += 1
                if os.path.exists(local_pq):
                    os.remove(local_pq)
            except Exception as err:
                print(f'[!] Gagal memproses {shard_path}: {err}')

    records = []
    for p in existing_ai[:n_ai]:
        records.append({'image_path': p, 'mask_path': None, 'label': 0, 'subset': 'CommunityForensics (AI)'})
    for p in existing_real[:n_real]:
        records.append({'image_path': p, 'mask_path': None, 'label': 1, 'subset': 'CommunityForensics (Real)'})
    return records

def crop_image_and_mask(img, mask_path, min_crop=320):
    try:
        mask = Image.open(mask_path).convert('L')
        if mask.size != img.size:
            mask = mask.resize(img.size, Image.Resampling.NEAREST)
        bbox = mask.point(lambda p: 255 if p > 15 else 0).getbbox()
        if not bbox:
            return img, mask
        x1, y1, x2, y2 = bbox
        w, h = img.size
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        win = min(max(min_crop, int(max(x2 - x1, y2 - y1) * 1.6)), max(w, h))
        half = win // 2
        left = max(0, min(w - win, cx - half)) if w > win else 0
        top = max(0, min(h - win, cy - half)) if h > win else 0
        box = (left, top, min(w, left + win), min(h, top + win))
        return img.crop(box), mask.crop(box)
    except Exception:
        return img, None

def load_defacto_inpainting_with_masks(n_samples=3000):
    CACHE_IMG = 'data/defacto_cache/inpainting'
    CACHE_MSK = 'data/defacto_cache/inpainting_masks'
    os.makedirs(CACHE_IMG, exist_ok=True)
    os.makedirs(CACHE_MSK, exist_ok=True)
    existing = sorted([os.path.join(CACHE_IMG, f) for f in os.listdir(CACHE_IMG) if f.lower().endswith(VALID_EXTS)])

    if len(existing) < n_samples:
        print(f'[*] Mengunduh Kaggle defactodataset/defactoinpainting (target: {n_samples})...')
        try:
            dl_path = kagglehub.dataset_download('defactodataset/defactoinpainting')
            candidates, mask_map = [], {}
            skip_keywords = ('mask', 'annot', 'graph', 'probe', 'donor')
            for root, _, files in os.walk(dl_path):
                root_lower = root.lower()
                is_mask_dir = ('probe_mask' in root_lower) or ('inpaint_mask' in root_lower) or ('mask' in root_lower and 'donor' not in root_lower)
                for f in files:
                    if not f.lower().endswith(VALID_EXTS):
                        continue
                    stem = os.path.splitext(f)[0]
                    full_p = os.path.join(root, f)
                    if is_mask_dir:
                        mask_map[stem] = full_p
                    elif not any(k in root_lower for k in skip_keywords) and not any(k in f.lower() for k in skip_keywords):
                        candidates.append((stem, full_p))
            random.shuffle(candidates)
            count = len(existing)
            for stem, src_p in tqdm(candidates, desc='Ekstrak DEFACTO Inpainting + Mask', leave=False):
                if count >= n_samples:
                    break
                try:
                    img = Image.open(src_p)
                    if img.mode in ('1', 'L', 'I', 'F'):
                        continue
                    img = img.convert('RGB')
                    msk = None
                    if stem in mask_map:
                        img, msk = crop_image_and_mask(img, mask_map[stem], min_crop=320)
                    if max(img.size) > 2048:
                        img.thumbnail((2048, 2048), Image.Resampling.BILINEAR)
                    save_p = f'{CACHE_IMG}/defacto_inpainting_{count:05d}.jpg'
                    save_m = f'{CACHE_MSK}/defacto_inpainting_{count:05d}.png'
                    if not os.path.exists(save_p):
                        img.save(save_p, 'JPEG', quality=random.randint(75, 98))
                    if msk is not None and not os.path.exists(save_m):
                        msk.resize(img.size, Image.Resampling.NEAREST).save(save_m, 'PNG')
                    existing.append(save_p)
                    count += 1
                except Exception:
                    continue
            if os.path.exists(dl_path):
                shutil.rmtree(dl_path, ignore_errors=True)
        except Exception as err:
            print(f'[!] Gagal mengunduh defactoinpainting: {err}')

    records = []
    for p in existing[:n_samples]:
        m_candidate = os.path.join(CACHE_MSK, os.path.splitext(os.path.basename(p))[0] + '.png')
        records.append({
            'image_path': p,
            'mask_path': m_candidate if os.path.exists(m_candidate) else None,
            'label': 0,
            'subset': 'DEFACTO (Inpainting)'
        })
    return records

all_records = load_community_forensics(CF_REAL_SAMPLES, CF_AI_SAMPLES) + load_defacto_inpainting_with_masks(DEFACTO_INPAINT_SAMPLES)
df_all = pd.DataFrame(all_records)
n_bal = min((df_all['label'] == 0).sum(), (df_all['label'] == 1).sum())
df_dataset = pd.concat([
    df_all[df_all['label'] == 0].sample(n=n_bal, random_state=42),
    df_all[df_all['label'] == 1].sample(n=n_bal, random_state=42)
]).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(df_dataset['subset'].value_counts())

In [ ]:
# @title 🔀 3. Dual-Task DataLoader (Citra 224x224 + Target Anomaly Mask 14x14)
IMG_SIZE = 224
MASK_SIZE = 14
BATCH_SIZE = 64

train_df, temp_df = train_test_split(df_dataset, test_size=0.20, random_state=42, stratify=df_dataset['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])
train_df, val_df, test_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

class DualForensicDataset(Dataset):
    """
    Menghasilkan (image_tensor, label, mask_14x14):
    - Real (label=1)          -> mask_14x14 bernilai 0.0 di seluruh piksel
    - Full AI (label=0, no mask) -> mask_14x14 bernilai 1.0 di seluruh piksel
    - Inpainting (label=0, ada mask) -> mask_14x14 sesuai area inpaint_mask yang di-crop sinkron dengan gambar
    """
    def __init__(self, df, is_train=True):
        self.paths = df['image_path'].tolist()
        self.masks = df['mask_path'].tolist()
        self.labels = df['label'].tolist()
        self.is_train = is_train
        self.norm = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        lbl = self.labels[idx]
        m_path = self.masks[idx]
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128, 128, 128))

        w, h = img.size
        if w < IMG_SIZE or h < IMG_SIZE:
            scale = max(IMG_SIZE / w, IMG_SIZE / h)
            img = img.resize((int(round(w * scale)), int(round(h * scale))), Image.Resampling.BILINEAR)
            w, h = img.size

        left = random.randint(0, w - IMG_SIZE) if self.is_train else (w - IMG_SIZE) // 2
        top = random.randint(0, h - IMG_SIZE) if self.is_train else (h - IMG_SIZE) // 2
        img_crop = img.crop((left, top, left + IMG_SIZE, top + IMG_SIZE))

        if m_path and os.path.exists(str(m_path)):
            try:
                msk = Image.open(m_path).convert('L').resize((w, h), Image.Resampling.NEAREST)
                msk_crop = msk.crop((left, top, left + IMG_SIZE, top + IMG_SIZE))
                msk_14 = msk_crop.resize((MASK_SIZE, MASK_SIZE), Image.Resampling.BILINEAR)
                target_mask = (torch.from_numpy(np.array(msk_14, dtype=np.float32)) / 255.0).unsqueeze(0).clamp(0.0, 1.0)
                # Jika random crop kebetulan meleset dari area inpaint, beri nilai dasar minimal 0.5 karena gambar tetap berasal dari kelas dimanipulasi
                if target_mask.max() < 0.1:
                    target_mask.fill_(0.5)
            except Exception:
                target_mask = torch.ones((1, MASK_SIZE, MASK_SIZE), dtype=torch.float32)
        else:
            target_mask = torch.zeros((1, MASK_SIZE, MASK_SIZE), dtype=torch.float32) if lbl == 1 else torch.ones((1, MASK_SIZE, MASK_SIZE), dtype=torch.float32)

        if self.is_train:
            if random.random() < 0.5:
                img_crop = img_crop.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
                target_mask = torch.flip(target_mask, dims=[2])
            if random.random() < 0.4:
                buf = io.BytesIO()
                img_crop.save(buf, format='JPEG', quality=random.randint(70, 95))
                buf.seek(0)
                img_crop = Image.open(buf).convert('RGB')

        return self.norm(img_crop), torch.tensor(lbl, dtype=torch.long), target_mask

train_loader = DataLoader(DualForensicDataset(train_df, is_train=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(DualForensicDataset(val_df, is_train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(DualForensicDataset(test_df, is_train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Split -> Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

In [ ]:
# @title 🧠 4. Arsitektur AnomalyForensicNet (Clean-Slate From-Scratch)
class LayerNorm2dClean(nn.Module):
    def __init__(self, channels, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(channels))
        self.bias = nn.Parameter(torch.zeros(channels))
        self.eps = eps
    def forward(self, x):
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        return self.weight[:, None, None] * ((x - u) / torch.sqrt(s + self.eps)) + self.bias[:, None, None]

class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        reduced = max(8, channels // reduction)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, reduced, bias=False), nn.GELU(),
            nn.Linear(reduced, channels, bias=False), nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        return x * self.fc(x).view(b, c, 1, 1)

class NPRModule(nn.Module):
    def forward(self, x):
        down = F.interpolate(x, scale_factor=0.5, mode='nearest')
        return x - F.interpolate(down, size=x.shape[2:], mode='nearest')

class MultiScaleForensicFilterBankV4(nn.Module):
    def __init__(self):
        super().__init__()
        lap3 = torch.tensor([[-1., -1., -1.], [-1., 8., -1.], [-1., -1., -1.]])
        lap5 = torch.tensor([[0., 0., -1., 0., 0.], [0., -1., -2., -1., 0.], [-1., -2., 16., -2., -1.], [0., -1., -2., -1., 0.], [0., 0., -1., 0., 0.]])
        dog = torch.tensor([[-1., -3., -4., -3., -1.], [-3., 0., 6., 0., -3.], [-4., 6., 20., 6., -4.], [-3., 0., 6., 0., -3.], [-1., -3., -4., -3., -1.]]) / 16.0
        sobel_h = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]])
        sobel_v = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
        self.register_buffer('k3', torch.stack([lap3, sobel_h, sobel_v]).unsqueeze(1).repeat(3, 1, 1, 1))
        self.register_buffer('k5', torch.stack([lap5, dog]).unsqueeze(1).repeat(3, 1, 1, 1))
        self.npr = NPRModule()
    def forward(self, x):
        f3 = F.conv2d(x, self.k3, padding=1, groups=3)
        f5 = F.conv2d(x, self.k5, padding=2, groups=3)
        smag = torch.sqrt(f3[:, 1::3] ** 2 + f3[:, 2::3] ** 2 + 1e-6)
        return torch.cat([f3[:, 0::3], f5[:, 0::2], f5[:, 1::2], smag, self.npr(x)], dim=1)

class ZeroMeanConv2d(nn.Module):
    """Constrained Conv2d (sum of weights = 0, center = -1) untuk menekan objek & belajar residu baru."""
    def __init__(self, in_c=3, out_c=9, kernel_size=5):
        super().__init__()
        self.padding = kernel_size // 2
        self.c = kernel_size // 2
        self.weight = nn.Parameter(torch.randn(out_c, in_c, kernel_size, kernel_size))
        nn.init.xavier_uniform_(self.weight)
    def forward(self, x):
        mask = torch.ones_like(self.weight)
        mask[:, :, self.c, self.c] = 0.0
        w_masked = self.weight * mask
        w_norm = w_masked / (w_masked.sum(dim=(2, 3), keepdim=True) + 1e-7)
        center = torch.zeros_like(self.weight)
        center[:, :, self.c, self.c] = 1.0
        return F.conv2d(x, w_norm * mask - center, padding=self.padding)

class PhaseMagSpectralStem(nn.Module):
    """Mengekstrak Log-Magnitude DAN Phase Spectrum 2D-FFT (peka terhadap kerusakan fase inpainting)."""
    def __init__(self, in_c=3, out_c=32):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(in_c * 2, out_c, kernel_size=3, stride=2, padding=1, bias=False),
            LayerNorm2dClean(out_c), nn.GELU()
        )
    def forward(self, x):
        fft = torch.fft.fftshift(torch.fft.fft2(x.float(), norm='ortho'), dim=(-2, -1))
        spec = torch.cat([torch.log1p(torch.abs(fft)), torch.angle(fft) / 3.14159265], dim=1).to(x.dtype)
        return self.proj(spec)

class MixKernelForensicBlock(nn.Module):
    """Mix-Kernel 3x3 (tepi jahitan inpainting) + 7x7 (periodisitas difusi) dengan LayerNorm2d."""
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.dw3 = nn.Conv2d(in_c, in_c, 3, stride=stride, padding=1, groups=in_c, bias=False)
        self.dw7 = nn.Conv2d(in_c, in_c, 7, stride=stride, padding=3, groups=in_c, bias=False)
        self.norm = LayerNorm2dClean(in_c)
        self.pw1 = nn.Conv2d(in_c, out_c * 2, 1)
        self.act = nn.GELU()
        self.pw2 = nn.Conv2d(out_c * 2, out_c, 1)
        self.se = SqueezeExcitation(out_c)
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False), LayerNorm2dClean(out_c)
        ) if stride != 1 or in_c != out_c else nn.Identity()
    def forward(self, x):
        feat = self.norm(self.dw3(x) + self.dw7(x))
        return self.se(self.pw2(self.act(self.pw1(feat)))) + self.shortcut(x)

class IntraImageInconsistencyAttention(nn.Module):
    """Menyorot patch yang menyimpang dari rata-rata statistik noise gambar itu sendiri."""
    def __init__(self, channels):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Conv2d(channels, channels // 4, 1), LayerNorm2dClean(channels // 4),
            nn.GELU(), nn.Conv2d(channels // 4, 1, 1), nn.Sigmoid()
        )
    def forward(self, x):
        diff = torch.abs(x - x.mean(dim=(2, 3), keepdim=True))
        return x * (1.0 + self.gate(diff))

class TopKAnomalyPool2d(nn.Module):
    """Mean + Std + Top-K (15%) Spatial Activation Pooling agar sinyal inpainting kecil tidak tenggelam."""
    def __init__(self, topk_ratio=0.15):
        super().__init__()
        self.topk_ratio = topk_ratio
    def forward(self, x):
        b, c, h, w = x.shape
        flat = x.view(b, c, h * w)
        k = max(1, int(round((h * w) * self.topk_ratio)))
        topk_mean = torch.topk(flat, k=k, dim=-1)[0].mean(dim=-1)
        return torch.cat([flat.mean(dim=-1), torch.sqrt(flat.var(dim=-1, unbiased=False) + 1e-6), topk_mean], dim=1)

class AnomalyForensicNet(nn.Module):
    def __init__(self, num_classes=2, dropout_p=0.3):
        super().__init__()
        self.fixed_bank = MultiScaleForensicFilterBankV4()
        self.zeromean_conv = ZeroMeanConv2d(3, 9, kernel_size=5)
        self.forensic_stem = nn.Sequential(nn.Conv2d(24, 32, 3, stride=2, padding=1, bias=False), LayerNorm2dClean(32), nn.GELU())
        self.spectral_stem = PhaseMagSpectralStem(3, 32)
        self.rgb_stem = nn.Sequential(nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False), LayerNorm2dClean(32), nn.GELU())
        self.fusion = nn.Sequential(nn.Conv2d(96, 64, 1, bias=False), LayerNorm2dClean(64), nn.GELU())

        self.stage1 = nn.Sequential(MixKernelForensicBlock(64, 64), MixKernelForensicBlock(64, 64))
        self.stage2 = nn.Sequential(MixKernelForensicBlock(64, 128, stride=2), MixKernelForensicBlock(128, 128))
        self.stage3 = nn.Sequential(MixKernelForensicBlock(128, 256, stride=2), MixKernelForensicBlock(256, 256))
        self.stage4 = nn.Sequential(MixKernelForensicBlock(256, 384, stride=2), MixKernelForensicBlock(384, 384))

        self.inconsistency_attn = IntraImageInconsistencyAttention(384)
        self.mask_head = nn.Sequential(
            nn.Conv2d(384, 64, 3, padding=1, bias=False), LayerNorm2dClean(64), nn.GELU(), nn.Conv2d(64, 1, 1)
        )
        self.pool = TopKAnomalyPool2d(topk_ratio=0.15)
        self.head = nn.Sequential(
            nn.Dropout(p=dropout_p), nn.Linear(384 * 3, 192), nn.LayerNorm(192),
            nn.GELU(), nn.Dropout(p=0.15), nn.Linear(192, num_classes)
        )

    def forward(self, x, return_mask=False):
        res_24 = torch.cat([self.fixed_bank(x), self.zeromean_conv(x)], dim=1)
        feat = self.fusion(torch.cat([self.forensic_stem(res_24), self.spectral_stem(x), self.rgb_stem(x)], dim=1))
        feat = self.inconsistency_attn(self.stage4(self.stage3(self.stage2(self.stage1(feat)))))
        logits = self.head(self.pool(feat))
        if return_mask:
            return logits, self.mask_head(feat)
        return logits

model = AnomalyForensicNet(num_classes=2).to(device)
BEST_MODEL_PATH = 'best_anomaly_forensicnet_cf_inpainting.pth'
print(f'AnomalyForensicNet Siap! Total Parameter: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# @title 🚀 5. Dual-Task Training (Focal Loss Klasifikasi + Auxiliary Mask Localization Loss)
EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 4e-4
MASK_LOSS_WEIGHT = 0.4

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.gamma, self.label_smoothing = gamma, label_smoothing
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, label_smoothing=self.label_smoothing, reduction='none')
        return (((1.0 - torch.exp(-ce)) ** self.gamma) * ce).mean()

cls_criterion = FocalLoss(gamma=2.0, label_smoothing=0.05)
mask_criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
use_amp = (device.type == 'cuda')
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

best_val_f1, no_improve = 0.0, 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for images, labels, target_masks in tqdm(train_loader, desc=f'Epoch {epoch:02d}/{EPOCHS:02d} [Train]', leave=False):
        images, labels, target_masks = images.to(device, non_blocking=True), labels.to(device, non_blocking=True), target_masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits, mask_logits = model(images, return_mask=True)
            loss = cls_criterion(logits, labels) + MASK_LOSS_WEIGHT * mask_criterion(mask_logits, target_masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)

    model.eval()
    val_loss, val_preds, val_targets = 0.0, [], []
    with torch.no_grad():
        for images, labels, target_masks in val_loader:
            images, labels, target_masks = images.to(device), labels.to(device), target_masks.to(device)
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits, mask_logits = model(images, return_mask=True)
                loss = cls_criterion(logits, labels) + MASK_LOSS_WEIGHT * mask_criterion(mask_logits, target_masks)
            val_loss += loss.item() * images.size(0)
            val_preds.extend(logits.argmax(dim=1).cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    val_f1 = f1_score(val_targets, val_preds, average='macro') * 100
    scheduler.step()
    saved = ''
    if val_f1 > best_val_f1:
        best_val_f1, no_improve = val_f1, 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        saved = '==> [SAVED]'
    else:
        no_improve += 1
    print(f'Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {running_loss/len(train_df):.4f} | Val Loss: {val_loss/len(val_df):.4f} | Val F1: {val_f1:.2f}% {saved}')
    if no_improve >= PATIENCE:
        print(f'Early stopping pada epoch {epoch}.')
        break

# Evaluasi Test Set + Visualisasi Anomaly Heatmap 14x14
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()
test_preds, test_targets, test_probs = [], [], []
with torch.no_grad():
    for images, labels, _ in tqdm(test_loader, desc='Test Set (TTA)'):
        images = images.to(device)
        with torch.cuda.amp.autocast(enabled=use_amp):
            probs = (torch.softmax(model(images), dim=1) + torch.softmax(model(torch.flip(images, dims=[3])), dim=1)) / 2.0
        test_probs.extend(probs[:, 1].cpu().numpy())
        test_preds.extend(probs.argmax(dim=1).cpu().numpy())
        test_targets.extend(labels.numpy())

print(classification_report(test_targets, test_preds, target_names=['AI / Inpainting (0)', 'Real (1)'], digits=4))
print(f'ROC-AUC: {roc_auc_score(test_targets, test_probs)*100:.2f}%')